# 🧬 Project 5 — Semantic Correspondence
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup
Eseguire per configurare Drive, clonare il repo e scaricare il dataset.
**Sicuro da rieseguire**: gestisce automaticamente cartella già esistente e dataset già scaricato.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Crea cartelle su Drive se non esistono
os.makedirs('/content/drive/MyDrive/AML/Checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/AML/Results', exist_ok=True)

# Clona o aggiorna il repo (sicuro da rieseguire)
if os.path.exists('/content/Project_AML'):
    %cd /content/Project_AML
    !git pull origin osagie5
else:
    !git clone -b osagie5 https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
    %cd /content/Project_AML

!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

## 🔍 1. Stage 1: Backbone Analysis

In [ ]:
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --num_workers 2 --results_file /content/drive/MyDrive/AML/Results/baseline.txt

In [ ]:
for layer in [4, 8, 11]:
    !python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --layer {layer} --num_workers 2 --results_file /content/drive/MyDrive/AML/Results/layer_{layer}.txt

## 🚀 2. Stage 2: Fine-Tuning Efficiente (LoRA)

In [ ]:
!python train.py --exp_name lora_only \
    --dataset_root ./data/SPair-71k \
    --epochs 5 \
    --curriculum_epochs 0 \
    --num_workers 2 \
    --output_dir ./checkpoints/lora_only \
    --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_only

In [ ]:
!python train.py --exp_name lora_curriculum \
    --dataset_root ./data/SPair-71k \
    --epochs 5 \
    --curriculum_epochs 2 \
    --num_workers 2 \
    --output_dir ./checkpoints/lora_curriculum \
    --backup_dir /content/drive/MyDrive/AML/Checkpoints/lora_curriculum

## 🎯 3. Stage 3: Raffinamento Adattivo

In [ ]:
CKPT = '/content/drive/MyDrive/AML/Checkpoints/lora_curriculum/lora_curriculum_best.pth'
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint {CKPT} --num_workers 2 --results_file /content/drive/MyDrive/AML/Results/s3_with_aw.txt
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint {CKPT} --no_adaptive_win --num_workers 2 --results_file /content/drive/MyDrive/AML/Results/s3_without_aw.txt

## 🌟 4. Stage 4: Valutazione ed Estensioni

In [ ]:
# Valutazione finale LoRA only
CKPT_LORA = '/content/drive/MyDrive/AML/Checkpoints/lora_only/lora_only_best.pth'
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint {CKPT_LORA} --num_workers 2 --results_file /content/drive/MyDrive/AML/Results/s4_lora_only.txt

# Valutazione finale LoRA + Curriculum
CKPT_CURR = '/content/drive/MyDrive/AML/Checkpoints/lora_curriculum/lora_curriculum_best.pth'
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint {CKPT_CURR} --num_workers 2 --results_file /content/drive/MyDrive/AML/Results/s4_lora_curriculum.txt